# Sweetviz EDA: Train vs Test

This notebook:

1. Loads `train.csv` and `test.csv`
2. Generates Sweetviz EDA reports for each dataset
3. Compares shared columns between train and test
4. Generates train-only Sweetviz reports using each candidate target column

**Outputs written to HTML** (in the working directory):
- `train_analysis.html`
- `test_analysis.html`
- `train_vs_test_sweetviz.html`
- `train_analysis_target_event.html`
- `train_analysis_target_time_to_hit_hours.html`


In [ ]:
# --- Setup ---
# (Optional) install/upgrade sweetviz. Set to True only if you need it.
RUN_PIP_INSTALL = False

if RUN_PIP_INSTALL:
    !pip install sweetviz --upgrade --quiet

import pandas as pd
import numpy as np
import sweetviz as sv

# Fix for some environments where Sweetviz expects this attribute
if not hasattr(np, "VisibleDeprecationWarning"):
    np.VisibleDeprecationWarning = DeprecationWarning

# --- Load data ---
TRAIN_PATH = "./../data/train.csv"
TEST_PATH = "./../data/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df.head()

## 1) Identify candidate target columns

We expect `train.csv` to contain one or more target columns that are **not** present in `test.csv` (since `test` is typically what we generate predictions for). The cells below:

- List columns that exist in train but not test (target candidates)
- Sanity-check if any columns exist in test but not train (usually should be empty)


In [ ]:
# Columns present in train but not in test (typically includes the target)
target_candidates = sorted(set(train_df.columns) - set(test_df.columns))

# Columns present in test but not in train (sanity check)
test_only_columns = sorted(set(test_df.columns) - set(train_df.columns))

target_candidates, test_only_columns

## 2) Generate Sweetviz reports

This section writes the main Sweetviz HTML reports to disk:

- Train EDA report
- Test EDA report
- Train vs Test comparison report (common columns only)


In [ ]:
# Main Sweetviz reports
train_analysis = sv.analyze(train_df)
train_analysis.show_html("train_analysis.html", open_browser=False)

test_analysis = sv.analyze(test_df)
test_analysis.show_html("test_analysis.html", open_browser=False)

common_cols = sorted(set(train_df.columns).intersection(set(test_df.columns)))
train_for_compare = train_df[common_cols]
test_for_compare = test_df[common_cols]

sweetviz_compare = sv.compare([train_for_compare, "Train"], [test_for_compare, "Test"])
sweetviz_compare.show_html("train_vs_test_sweetviz.html", open_browser=False)


In [ ]:
# Sweetviz analysis for train using each candidate target variable
for target in target_candidates:
    train_report = sv.analyze(train_df, target_feat=target)
    train_report.show_html(f"train_analysis_target_{target}.html", open_browser=False)
